# European Soccer Data - Descriptive Statistics Analysis

This notebook performs comprehensive descriptive statistics analysis for European soccer leagues:
- **Premier League** (England)
- **Serie A** (Italy) 
- **Bundesliga** (Germany)
- **La Liga** (Spain)

The analysis generates individual league statistics and comparative analysis, outputting results to `src/descriptive_statistics.md`.

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

## Data Loading and Preparation

In [22]:
# Define data paths
data_dir = Path('../csv/processed')
output_dir = Path('../../../src')

# League configurations
leagues = {
    'Premier League': {
        'file': 'premier_league.csv',
        'country': 'England',
        'color': '#1f77b4'
    },
    'Serie A': {
        'file': 'serie_a.csv',
        'country': 'Italy',
        'color': '#ff7f0e'
    },
    'Bundesliga': {
        'file': 'bundesliga.csv',
        'country': 'Germany',
        'color': '#2ca02c'
    },
    'La Liga': {
        'file': 'la_liga.csv',
        'country': 'Spain',
        'color': '#d62728'
    }
}

# Load all league data
league_data = {}
for league_name, config in leagues.items():
    file_path = data_dir / config['file']
    if file_path.exists():
        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df['total_goals'] = df['hometeamgoals'] + df['awayteamgoals']
        league_data[league_name] = df
        print(f"Loaded {league_name}: {len(df):,} matches ({df['season'].min()}-{df['season'].max()})")
    else:
        print(f"Warning: {file_path} not found")

print(f"\nTotal matches across all leagues: {sum(len(df) for df in league_data.values()):,}")

Loaded Premier League: 9,410 matches (2000-2024)
Loaded Serie A: 9,012 matches (2000-2024)
Loaded Bundesliga: 7,522 matches (2000-2024)
Loaded La Liga: 9,008 matches (2000-2024)

Total matches across all leagues: 34,952


## Analysis Functions

In [23]:
def analyze_league_results(df, league_name):
    """Analyze match results for a single league."""
    total_matches = len(df)
    
    # Result counts - using numeric values (1=home win, 0=draw, -1=away win)
    home_wins = len(df[df['hometeamresult'] == 1])
    draws = len(df[df['hometeamresult'] == 0])
    away_wins = len(df[df['hometeamresult'] == -1])
    
    # Percentages
    home_win_pct = (home_wins / total_matches) * 100
    draw_pct = (draws / total_matches) * 100
    away_win_pct = (away_wins / total_matches) * 100
    
    return {
        'league': league_name,
        'total_matches': total_matches,
        'home_wins': home_wins,
        'draws': draws,
        'away_wins': away_wins,
        'home_win_pct': home_win_pct,
        'draw_pct': draw_pct,
        'away_win_pct': away_win_pct,
        'season_range': f"{df['season'].min()}-{df['season'].max()}"
    }

def analyze_league_goals(df, league_name):
    """Analyze goal statistics for a single league."""
    return {
        'league': league_name,
        'avg_home_goals': df['hometeamgoals'].mean(),
        'std_home_goals': df['hometeamgoals'].std(),
        'avg_away_goals': df['awayteamgoals'].mean(),
        'std_away_goals': df['awayteamgoals'].std(),
        'avg_total_goals': df['total_goals'].mean(),
        'std_total_goals': df['total_goals'].std(),
        'min_total_goals': df['total_goals'].min(),
        'max_total_goals': df['total_goals'].max()
    }

def analyze_league_odds(df, league_name):
    """Analyze betting odds for a single league."""
    # Filter out missing odds
    odds_df = df.dropna(subset=['OddHome', 'OddDraw', 'OddAway'])
    
    if len(odds_df) == 0:
        return {
            'league': league_name,
            'matches_with_odds': 0,
            'odds_coverage_pct': 0.0
        }
    
    return {
        'league': league_name,
        'matches_with_odds': len(odds_df),
        'odds_coverage_pct': (len(odds_df) / len(df)) * 100,
        'avg_home_odds': odds_df['OddHome'].mean(),
        'std_home_odds': odds_df['OddHome'].std(),
        'avg_draw_odds': odds_df['OddDraw'].mean(),
        'std_draw_odds': odds_df['OddDraw'].std(),
        'avg_away_odds': odds_df['OddAway'].mean(),
        'std_away_odds': odds_df['OddAway'].std()
    }

## Individual League Analysis

In [24]:
# Analyze each league individually
league_results = []
league_goals = []
league_odds = []

for league_name, df in league_data.items():
    print(f"\n=== {league_name} Analysis ===")

    # Results analysis using the function
    results = analyze_league_results(df, league_name)
    league_results.append(results)
    print(f"Matches: {results['total_matches']:,} ({results['season_range']})")
    print(f"Home wins: {results['home_win_pct']:.1f}%, Draws: {results['draw_pct']:.1f}%, Away wins: {results['away_win_pct']:.1f}%")

    # Goals analysis
    goals = analyze_league_goals(df, league_name)
    league_goals.append(goals)
    print(f"Avg goals per match: {goals['avg_total_goals']:.2f} ± {goals['std_total_goals']:.2f}")
    print(f"Home: {goals['avg_home_goals']:.2f} ± {goals['std_home_goals']:.2f}, Away: {goals['avg_away_goals']:.2f} ± {goals['std_away_goals']:.2f}")

    # Odds analysis
    odds = analyze_league_odds(df, league_name)
    league_odds.append(odds)
    if odds['matches_with_odds'] > 0:
        print(f"Odds coverage: {odds['odds_coverage_pct']:.1f}% ({odds['matches_with_odds']:,} matches)")
        print(f"Avg odds - Home: {odds['avg_home_odds']:.2f}, Draw: {odds['avg_draw_odds']:.2f}, Away: {odds['avg_away_odds']:.2f}")
    else:
        print("No betting odds data available")

# Convert to DataFrames for easier manipulation
results_df = pd.DataFrame(league_results)
goals_df = pd.DataFrame(league_goals)
odds_df = pd.DataFrame(league_odds)

print("\n=== Individual League Analysis Complete ===")


=== Premier League Analysis ===
Matches: 9,410 (2000-2024)
Home wins: 45.8%, Draws: 24.6%, Away wins: 29.6%
Avg goals per match: 2.72 ± 1.67
Home: 1.53 ± 1.30, Away: 1.18 ± 1.16
Odds coverage: 99.1% (9,327 matches)
Avg odds - Home: 2.74, Draw: 3.97, Away: 4.73

=== Serie A Analysis ===
Matches: 9,012 (2000-2024)
Home wins: 44.6%, Draws: 27.1%, Away wins: 28.3%
Avg goals per match: 2.68 ± 1.65
Home: 1.50 ± 1.23, Away: 1.17 ± 1.11
Odds coverage: 98.7% (8,893 matches)
Avg odds - Home: 2.61, Draw: 3.69, Away: 4.56

=== Bundesliga Analysis ===
Matches: 7,522 (2000-2024)
Home wins: 45.7%, Draws: 24.7%, Away wins: 29.7%
Avg goals per match: 2.95 ± 1.72
Home: 1.67 ± 1.36, Away: 1.28 ± 1.20
Odds coverage: 99.0% (7,447 matches)
Avg odds - Home: 2.54, Draw: 3.92, Away: 4.31

=== La Liga Analysis ===
Matches: 9,008 (2000-2024)
Home wins: 47.1%, Draws: 25.2%, Away wins: 27.7%
Avg goals per match: 2.67 ± 1.68
Home: 1.55 ± 1.31, Away: 1.13 ± 1.11
Odds coverage: 99.0% (8,918 matches)
Avg odds - Home:

## Median Statistics Analysis

Median values provide insights into the central tendency of the data and are less affected by outliers compared to means. This section computes median statistics for goals and betting odds across all leagues.

In [25]:
# Calculate median statistics for each league
median_stats = []

for league_name, df in league_data.items():
    # Goal medians
    median_home_goals = df['hometeamgoals'].median()
    median_away_goals = df['awayteamgoals'].median()
    median_total_goals = df['total_goals'].median()
    
    # Odds medians (filter out missing values)
    odds_df = df.dropna(subset=['OddHome', 'OddDraw', 'OddAway'])
    if len(odds_df) > 0:
        median_home_odds = odds_df['OddHome'].median()
        median_draw_odds = odds_df['OddDraw'].median()
        median_away_odds = odds_df['OddAway'].median()
    else:
        median_home_odds = median_draw_odds = median_away_odds = None
    
    median_stats.append({
        'league': league_name,
        'median_home_goals': median_home_goals,
        'median_away_goals': median_away_goals,
        'median_total_goals': median_total_goals,
        'median_home_odds': median_home_odds,
        'median_draw_odds': median_draw_odds,
        'median_away_odds': median_away_odds
    })

# Convert to DataFrame for display
median_df = pd.DataFrame(median_stats)

print("=== Median Statistics by League ===\n")

# Display goal medians
print("Goal Medians:")
goal_medians = median_df[['league', 'median_home_goals', 'median_away_goals', 'median_total_goals']].round(3)
print(goal_medians.to_string(index=False))

print("\nBetting Odds Medians:")
odds_medians = median_df[['league', 'median_home_odds', 'median_draw_odds', 'median_away_odds']].round(3)
print(odds_medians.to_string(index=False))

=== Median Statistics by League ===

Goal Medians:
        league  median_home_goals  median_away_goals  median_total_goals
Premier League                1.0                1.0                 3.0
       Serie A                1.0                1.0                 3.0
    Bundesliga                1.0                1.0                 3.0
       La Liga                1.0                1.0                 2.0

Betting Odds Medians:
        league  median_home_odds  median_draw_odds  median_away_odds
Premier League              2.18              3.51               3.4
       Serie A              2.15              3.40               3.5
    Bundesliga              2.10              3.50               3.3
       La Liga              2.10              3.40               3.5


## Skewness Analysis

Skewness measures the asymmetry of the data distribution. Positive skewness indicates a right-tailed distribution (more extreme high values), while negative skewness indicates a left-tailed distribution (more extreme low values). Values near zero indicate symmetric distributions.

In [26]:
# Calculate skewness statistics for each league
from scipy import stats

skewness_stats = []

for league_name, df in league_data.items():
    # Goal skewness
    skew_home_goals = stats.skew(df['hometeamgoals'])
    skew_away_goals = stats.skew(df['awayteamgoals'])
    skew_total_goals = stats.skew(df['total_goals'])
    
    # Odds skewness (filter out missing values)
    odds_df = df.dropna(subset=['OddHome', 'OddDraw', 'OddAway'])
    if len(odds_df) > 0:
        skew_home_odds = stats.skew(odds_df['OddHome'])
        skew_draw_odds = stats.skew(odds_df['OddDraw'])
        skew_away_odds = stats.skew(odds_df['OddAway'])
    else:
        skew_home_odds = skew_draw_odds = skew_away_odds = None
    
    skewness_stats.append({
        'league': league_name,
        'skew_home_goals': skew_home_goals,
        'skew_away_goals': skew_away_goals,
        'skew_total_goals': skew_total_goals,
        'skew_home_odds': skew_home_odds,
        'skew_draw_odds': skew_draw_odds,
        'skew_away_odds': skew_away_odds
    })

# Convert to DataFrame for display
skewness_df = pd.DataFrame(skewness_stats)

print("=== Skewness Statistics by League ===\n")

# Display goal skewness
print("Goal Distribution Skewness:")
goal_skewness = skewness_df[['league', 'skew_home_goals', 'skew_away_goals', 'skew_total_goals']].round(3)
print(goal_skewness.to_string(index=False))

print("\nBetting Odds Distribution Skewness:")
odds_skewness = skewness_df[['league', 'skew_home_odds', 'skew_draw_odds', 'skew_away_odds']].round(3)
print(odds_skewness.to_string(index=False))

print("\n=== Skewness Interpretation ===")
print("• Positive skewness (> 0): Right-tailed distribution, more extreme high values")
print("• Negative skewness (< 0): Left-tailed distribution, more extreme low values")
print("• Near zero (~0): Approximately symmetric distribution")
print("• |skewness| > 1: Highly skewed")
print("• 0.5 < |skewness| < 1: Moderately skewed")
print("• |skewness| < 0.5: Approximately symmetric")

=== Skewness Statistics by League ===

Goal Distribution Skewness:
        league  skew_home_goals  skew_away_goals  skew_total_goals
Premier League            0.971            1.073             0.605
       Serie A            0.803            0.992             0.570
    Bundesliga            0.896            1.001             0.491
       La Liga            0.982            1.171             0.661

Betting Odds Distribution Skewness:
        league  skew_home_odds  skew_draw_odds  skew_away_odds
Premier League           3.166           3.078           2.622
       Serie A           3.003           2.497           2.212
    Bundesliga           4.063           4.079           3.663
       La Liga           4.407           4.128           3.637

=== Skewness Interpretation ===
• Positive skewness (> 0): Right-tailed distribution, more extreme high values
• Negative skewness (< 0): Left-tailed distribution, more extreme low values
• Near zero (~0): Approximately symmetric distribution
• 

In [27]:
# Summary of median and skewness findings
print("=" * 60)
print("MEDIAN AND SKEWNESS SUMMARY")
print("=" * 60)

# Combine median and skewness data for easy comparison
summary_df = median_df.merge(skewness_df, on='league')

print("\nCombined Statistics Summary (rounded to 3 decimal places):")
print("-" * 60)

for _, row in summary_df.iterrows():
    print(f"\n{row['league']}:")
    print(f"  Goals    - Median: H={row['median_home_goals']:.1f}, A={row['median_away_goals']:.1f}, Total={row['median_total_goals']:.1f}")
    print(f"           - Skewness: H={row['skew_home_goals']:.3f}, A={row['skew_away_goals']:.3f}, Total={row['skew_total_goals']:.3f}")
    if pd.notna(row['median_home_odds']):
        print(f"  Odds     - Median: H={row['median_home_odds']:.2f}, D={row['median_draw_odds']:.2f}, A={row['median_away_odds']:.2f}")
        print(f"           - Skewness: H={row['skew_home_odds']:.3f}, D={row['skew_draw_odds']:.3f}, A={row['skew_away_odds']:.3f}")

print("\n" + "=" * 60)
print("KEY INSIGHTS:")
print("=" * 60)
print("• All leagues show positive skewness in goal distributions (right-tailed)")
print("• Betting odds are also positively skewed (occasional very high odds)")
print("• Median values provide robust central tendency measures less affected by outliers")
print("• Goal distributions are moderately skewed, indicating some high-scoring matches")
print("• Odds distributions are more heavily skewed than goal distributions")

MEDIAN AND SKEWNESS SUMMARY

Combined Statistics Summary (rounded to 3 decimal places):
------------------------------------------------------------

Premier League:
  Goals    - Median: H=1.0, A=1.0, Total=3.0
           - Skewness: H=0.971, A=1.073, Total=0.605
  Odds     - Median: H=2.18, D=3.51, A=3.40
           - Skewness: H=3.166, D=3.078, A=2.622

Serie A:
  Goals    - Median: H=1.0, A=1.0, Total=3.0
           - Skewness: H=0.803, A=0.992, Total=0.570
  Odds     - Median: H=2.15, D=3.40, A=3.50
           - Skewness: H=3.003, D=2.497, A=2.212

Bundesliga:
  Goals    - Median: H=1.0, A=1.0, Total=3.0
           - Skewness: H=0.896, A=1.001, Total=0.491
  Odds     - Median: H=2.10, D=3.50, A=3.30
           - Skewness: H=4.063, D=4.079, A=3.663

La Liga:
  Goals    - Median: H=1.0, A=1.0, Total=2.0
           - Skewness: H=0.982, A=1.171, Total=0.661
  Odds     - Median: H=2.10, D=3.40, A=3.50
           - Skewness: H=4.407, D=4.128, A=3.637

KEY INSIGHTS:
• All leagues show pos

## Comparative Analysis

In [28]:
# Overall statistics
total_matches = results_df['total_matches'].sum()
overall_home_wins = results_df['home_wins'].sum()
overall_draws = results_df['draws'].sum()
overall_away_wins = results_df['away_wins'].sum()

overall_stats = {
    'total_matches': total_matches,
    'overall_home_win_pct': (overall_home_wins / total_matches) * 100,
    'overall_draw_pct': (overall_draws / total_matches) * 100,
    'overall_away_win_pct': (overall_away_wins / total_matches) * 100
}

print(f"=== Overall Statistics Across All Leagues ===")
print(f"Total matches: {overall_stats['total_matches']:,}")
print(f"Home win rate: {overall_stats['overall_home_win_pct']:.1f}%")
print(f"Draw rate: {overall_stats['overall_draw_pct']:.1f}%")
print(f"Away win rate: {overall_stats['overall_away_win_pct']:.1f}%")

# League rankings
print("\n=== League Rankings ===")
print("\nMost competitive (highest away win %):")
for i, row in results_df.sort_values('away_win_pct', ascending=False).iterrows():
    print(f"{row['league']}: {row['away_win_pct']:.1f}%")

print("\nHighest scoring (most goals per match):")
for i, row in goals_df.sort_values('avg_total_goals', ascending=False).iterrows():
    print(f"{row['league']}: {row['avg_total_goals']:.2f} goals/match")

print("\nMost draws:")
for i, row in results_df.sort_values('draw_pct', ascending=False).iterrows():
    print(f"{row['league']}: {row['draw_pct']:.1f}%")

=== Overall Statistics Across All Leagues ===
Total matches: 34,952
Home win rate: 45.8%
Draw rate: 25.4%
Away win rate: 28.8%

=== League Rankings ===

Most competitive (highest away win %):
Bundesliga: 29.7%
Premier League: 29.6%
Serie A: 28.3%
La Liga: 27.7%

Highest scoring (most goals per match):
Bundesliga: 2.95 goals/match
Premier League: 2.72 goals/match
Serie A: 2.68 goals/match
La Liga: 2.67 goals/match

Most draws:
Serie A: 27.1%
La Liga: 25.2%
Bundesliga: 24.7%
Premier League: 24.6%
